# Azure AI Agent with Observability Workshop

Learn how to add OpenTelemetry tracing to Azure AI Foundry agents using **manual tracing spans**. This approach works with all agents, including those with complex tools like Bing search.

## What You'll Learn
1. Configure Azure Monitor with Application Insights
2. Create custom OpenTelemetry spans
3. Add attributes to spans for better observability
4. **View metrics in the Foundry Monitor dashboard**
5. **View detailed traces in the Foundry Tracing view**
6. Optionally explore advanced analytics in Azure Portal
7. Handle real-world limitations (complex tool parameters)

## Prerequisites
- Azure AI Project with Application Insights configured
- An existing agent in Azure AI Foundry (we'll use "Travel-agent")
- `AZURE_AI_PROJECT_ENDPOINT` environment variable set
- Azure CLI authentication (`az login`)
- Python packages: `agent-framework`, `azure-monitor-opentelemetry`

## Workshop Flow

Follow these steps in order:

1. **Run Imports** - Load all required libraries
2. **Understand Code** - Review the tracing function
3. **Execute Function** - Run the agent and see tracing in action
4. **View in Foundry** - Check both Monitor tab (metrics) and Tracing tab (detailed spans)
5. **Learn Concepts** - Understand manual vs automatic tracing
6. **Optional: Azure Portal** - Explore advanced analytics with Application Insights

**⏱️ Time**: 20-25 minutes | **Level**: Intermediate

## Quick Reference

**Key Code Patterns You'll Learn:**

```python
# 1. Configure Azure Monitor
configure_azure_monitor(connection_string=conn_string, ...)

# 2. Create a span
with get_tracer().start_as_current_span("Operation Name") as span:
    span.set_attribute("key", "value")
    # Your code here

# 3. Get trace ID for correlation
trace_id = format_trace_id(span.get_span_context().trace_id)
```

**Where to View Your Traces:**

| Location | What You See | When to Use |
|----------|--------------|-------------|
| **Foundry Monitor Tab** | Token usage, latency, success rates | Operations, health checks |
| **Foundry Tracing Tab** | Span tree, custom attributes, timing | Debugging, development |
| **Azure Portal** | KQL queries, custom dashboards | Advanced analytics |

**Important**: This notebook uses **manual tracing** (not automatic instrumentation) to avoid serialization issues with complex tool parameters like Bing search.

---
## Exercise: Set Up Agent Monitoring with Application Insights
**Goal:** Connect Application Insights to monitor your agent from the Monitor tab

### Step 1: Navigate to Monitor Tab

**Action Steps:**
1. Look at the **top navigation bar** in the Foundry portal (at the very top of the page)
2. Click on the **"Monitor"** tab
3. You'll see the monitoring interface for your project

✅ **Checkpoint:** You're in the Monitor tab

### Step 2: Open Settings and Connect Application Insights

**Action Steps:**
1. In the **"Monitor"** tab, click **"Settings"** in the top right of the screen
2. In the Settings panel, find the **"Application insights resource name"** dropdown
3. Click **"Create new resource"** in the dropdown
4. In the creation dialog, provide:
   - **Name**: `foundry-agent-monitoring` (or your preferred unique name)
   - **Region**: Should auto-match your Foundry project region
   - **Log Analytics Workspace**: A default workspace will be auto-created and used
5. Click **"Create"**
6. Wait for the resource to be provisioned (30-60 seconds)

✅ **Checkpoint:** Application Insights resource is created and connected to your project

**What happens now?**
With monitoring in place, every agent run will automatically send telemetry data to Application Insights. You'll be able to view all this data in the Monitor tab after you interact with your agent.

---

In [ ]:
# Copyright (c) Microsoft. All rights reserved.

import asyncio
import logging
import os
import time

from agent_framework.azure import AzureAIProjectAgentProvider
from agent_framework.observability import create_resource, get_tracer
from azure.ai.projects.aio import AIProjectClient
from azure.identity.aio import AzureCliCredential
from azure.monitor.opentelemetry import configure_azure_monitor
from dotenv import load_dotenv
from opentelemetry.trace import SpanKind
from opentelemetry.trace.span import format_trace_id

load_dotenv()

# Enable nested asyncio for Jupyter notebooks
import nest_asyncio
nest_asyncio.apply()

# Set up logger
logger = logging.getLogger(__name__)

print("✅ All imports loaded successfully")
print("✅ Ready to run the workshop!")

## Step 1: Understanding the Code

The cell below defines a function that:
1. Connects to your Azure AI Project
2. Configures Azure Monitor tracing
3. Gets an existing agent from Azure AI Foundry
4. Creates **manual tracing spans** for observability
5. Asks the agent multiple questions and tracks them

**Key Concept**: We use manual tracing (`get_tracer().start_as_current_span()`) instead of automatic instrumentation to avoid serialization issues with complex tool parameters like Bing search.

### 📝 Code Walkthrough

Look for these key lines in the code below:

1. **Line ~23**: `configure_azure_monitor()` - Sets up connection to Application Insights
2. **Line ~73**: `get_tracer().start_as_current_span("Travel Agent Chat")` - Creates parent span
3. **Line ~81**: `current_span.set_attribute()` - Adds custom metadata
4. **Line ~87**: Child spans for each question - Creates span hierarchy
5. **Line ~93**: More custom attributes for each question

Run this cell to define the function (you'll see execution count increase):

In [ ]:
async def using_provider_get_agent() -> None:
    """Get an existing Azure AI agent and interact with it with tracing."""
    print("=== Get existing Azure AI agent with provider.get_agent() ===\n")

    # Create the client
    async with (
        AzureCliCredential() as credential,
        AIProjectClient(
            endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"], 
            credential=credential
        ) as project_client,
    ):
        # Configure Azure Monitor tracing
        # Note: We configure Azure Monitor but DON'T enable automatic agent instrumentation
        # because it tries to serialize tool parameters that aren't JSON serializable.
        # Instead, we'll use manual tracing with get_tracer().start_as_current_span()
        conn_string = None
        
        try:
            conn_string = await project_client.telemetry.get_application_insights_connection_string()
            print(f"✅ Application Insights connection string retrieved")
        except Exception as e:
            logger.warning(
                "No Application Insights connection string found for the Azure AI Project. "
                "Please ensure Application Insights is configured in your Azure AI project."
            )
            print(f"⚠️ Warning: {e}")
            print("Continuing without tracing...")
    
        if conn_string:
            try:
                configure_azure_monitor(
                    connection_string=conn_string,
                    enable_live_metrics=True,
                    resource=create_resource(),
                    enable_performance_counters=False,
                )
                # NOTE: We do NOT call enable_instrumentation() here because it would
                # automatically instrument agent.run() calls and try to serialize tool
                # parameters, causing "BingGroundingSearchToolParameters is not JSON serializable"
                # Instead, we use manual tracing with get_tracer().start_as_current_span()
                print("✅ Azure Monitor configured for tracing")
                print("✅ Using manual tracing spans (not automatic agent instrumentation)\n")
                print("ℹ️ Note: Automatic agent instrumentation is disabled to avoid")
                print("   serialization issues with complex tool parameters like Bing search.")
                print("   You'll still see traces in Application Insights for the overall flow.\n")
            except Exception as e:
                logger.error(f"Failed to configure Azure Monitor: {e}")
                print(f"⚠️ Failed to configure Azure Monitor: {e}")
                print("Continuing without tracing...")
                conn_string = None
        
        # Get existing agent by name
        provider = AzureAIProjectAgentProvider(project_client=project_client)
        agent = await provider.get_agent(name="Travel-agent")  # Replace with your actual agent name

        # Verify agent properties
        print(f"Agent ID: {agent.id}")
        print(f"Agent name: {agent.name}")
        print(f"Agent description: {agent.description}\n")

        # List of questions to ask
        questions = [
            "Which destination fits a relaxed food-focused traveler?",
            "What festivals are happening in Rio this month?",
            "I want to visit Paris. What persona best meets this location? Also, what is the closest hotel to the Eiffel Tower?",
        ]

        # Create a manual tracing span for the agent interaction
        # This allows us to trace the overall flow without the automatic agent instrumentation
        # that fails on complex tool parameter serialization
        with get_tracer().start_as_current_span("Travel Agent Chat", kind=SpanKind.CLIENT) as current_span:
            if conn_string:
                trace_id = format_trace_id(current_span.get_span_context().trace_id)
                print(f"📊 Trace ID: {trace_id}")
                print("   Use this Trace ID to find traces in Application Insights\n")
                current_span.set_attribute("agent.id", agent.id)
                current_span.set_attribute("agent.name", agent.name)
                current_span.set_attribute("question.count", len(questions))
            
            # Ask each question with a delay
            for i, query in enumerate(questions, 1):
                # Create a span for each individual question
                with get_tracer().start_as_current_span(f"Question {i}", kind=SpanKind.CLIENT) as question_span:
                    if conn_string:
                        question_span.set_attribute("question.number", i)
                        question_span.set_attribute("question.text", query)
                    
                    print(f"\n{'='*60}")
                    print(f"Question {i} of {len(questions)}")
                    print(f"{'='*60}")
                    print(f"User: {query}")
                    print()
                    
                    try:
                        result = await agent.run(query)
                        print(f"Agent: {result}")
                        
                        if conn_string:
                            question_span.set_attribute("response.received", True)
                            question_span.set_attribute("response.length", len(str(result)))
                    except Exception as e:
                        print(f"❌ Error running agent: {e}")
                        if conn_string:
                            question_span.set_attribute("error", True)
                            question_span.set_attribute("error.type", type(e).__name__)
                            question_span.set_attribute("error.message", str(e))
                        raise
                
                # Wait 5 seconds before next question (except after the last one)
                if i < len(questions):
                    print("\n⏳ Waiting 5 seconds before next question...")
                    time.sleep(5)
        
        print("\n✅ All questions completed")
        if conn_string:
            print("📊 Check Application Insights for detailed traces and metrics")
            print("   You should see:")
            print("   - A 'Travel Agent Chat' span containing all questions")
            print("   - Individual 'Question N' spans for each question")
            print("   - Custom attributes like question text and response length")

## Step 2: Run the Agent with Tracing

Now execute the function to see tracing in action!

**What to expect:**
- ✅ Confirmation that Azure Monitor is configured
- 📊 A unique **Trace ID** (save this for later reference)
- 🤖 Agent responses to 3 travel questions
- ⏳ 5-second delays between questions

**After this runs**, the telemetry will flow to Application Insights and appear in:
- **Foundry Monitor tab** - Aggregate metrics (token usage, latency, success rate)
- **Foundry Tracing tab** - Detailed traces with span hierarchy and custom attributes
- **Azure Portal Application Insights** - Advanced analytics (optional)

**Note:** If you modified the agent name from "Travel-agent", update it in the function code above before running this cell.

In [ ]:
# Run the agent interaction with tracing
await using_provider_get_agent()

## Step 3: View Your Traces in Microsoft Foundry

After running your agent, view the telemetry in the Foundry portal. There are two key views:

### 📊 Option A: Monitor Tab (Aggregate Metrics)

**Best for**: Understanding overall performance, trends, and operational health

1. Open [Microsoft Foundry](https://ai.azure.com) (make sure the "New Foundry" toggle is on)
2. Navigate to the **Build** page using the top navigation
3. Select your agent (e.g., "Travel-agent")
4. Click the **Monitor** tab

**What You'll See:**
- **Charts**: Agents runs, token metrics, tool calls, error rates
- **Time Range**: Adjust to "Last hour" or "Last 4 hours" to see your recent runs

### 🔍 Option B: Tracing Tab (Detailed Traces)

**Best for**: Debugging specific runs, viewing custom spans and attributes

1. In [Microsoft Foundry](https://ai.azure.com), navigate to the **Tracing** tab in the top navigation pane
2. Filter traces by time range, agent name, or other criteria
3. Find your trace by looking for recent activity or search by operation name
4. **Click on a trace** to view the detailed span hierarchy

**What You'll See:**
- **Span Tree**: The parent "invoke_agent..." span and child "Question N" spans we created
- **Each Span Shows**:
  - Duration and timing
  - Custom attributes we added (`agent.id`, `question.text`, `response.length`, etc.)
  - Input/output information
  - Any errors or exceptions
- **Navigation**: Step through each span to identify bottlenecks or issues

**💡 Pro Tip**: The Tracing tab is perfect for understanding exactly what your manual tracing implementation did - you can see all the custom spans and attributes we created!

## Bonus: Additional Questions to Try

Want to experiment more? Here are additional questions you can add to the `questions` list in the function above:

In [ ]:
# Example: Additional questions you can add to the list
additional_questions = [
    "Tell me about things to do in New York.",
    "What is the best destination for a family vacation of 5?",
    "Recommend a destination for a traveler who enjoys nature, scenery, and light physical activity.",
    "Which destination fits a high-energy traveler who wants iconic city experiences without burnout?",
    "For Barcelona, create a food-focused itinerary with minimal transit.",
]

print("Additional questions you can try:")
for i, q in enumerate(additional_questions, 1):
    print(f"{i}. {q}")

---
## 🎉 Congratulations!

You've successfully completed the Azure AI Agent Observability workshop! You now have hands-on experience with production-ready monitoring for AI agents.

### What You Learned

✅ **Set up Application Insights** - Connected monitoring to your Azure AI Project  
✅ **Configured Azure Monitor** - Integrated OpenTelemetry tracing with your agent  
✅ **Implemented Manual Tracing** - Created custom spans to track agent behavior  
✅ **Added Custom Attributes** - Enriched traces with meaningful metadata  
✅ **Built Span Hierarchies** - Organized traces with parent-child relationships  
✅ **Monitored in Foundry** - Viewed metrics in the Monitor tab and traces in the Tracing tab  
✅ **Handled Real-World Constraints** - Worked around serialization issues with complex tools  
✅ **Gained Production Skills** - Learned observability patterns used in production environments

### Resources

**Documentation:**
- [Azure Monitor OpenTelemetry](https://learn.microsoft.com/azure/azure-monitor/app/opentelemetry-enable) - OpenTelemetry integration guide
- [Application Insights](https://learn.microsoft.com/azure/azure-monitor/app/app-insights-overview) - Monitoring documentation
- [Azure AI Foundry Monitoring](https://learn.microsoft.com/azure/ai-foundry/observability/how-to/how-to-monitor-agents-dashboard) - Agent monitoring guide
- [OpenTelemetry Python SDK](https://opentelemetry.io/docs/languages/python/) - OpenTelemetry documentation

**Related Lessons:**
- [Lesson 10: AI Agents in Production](../../10-ai-agents-production/README.md) - Production deployment strategies
- [Lesson 06: Building Trustworthy Agents](../../06-building-trustworthy-agents/README.md) - Safety and reliability
- [Lesson 14: Microsoft Agent Framework](../../14-microsoft-agent-framework/README.md) - Framework deep dive

**Community:**
- [Azure AI Foundry Discord](https://aka.ms/ai-agents/discord) - Get help and share learnings
- [Workshop Issues](https://github.com/microsoft/ai-agents-for-beginners/issues) - Report issues or suggest improvements